# 🎵 Spotify Hit Prediction — ML vs ANN Showdown
## Can AI Predict the Next Chart-Topper?
---
**Domain:** Entertainment (Music Streaming)  
**Use Case:** Predict if a song will be a **hit** (top 45% popularity) based on audio features  
**Dataset:** 12,000 Spotify-style tracks with 14 audio features  
**Task:** Binary Classification —  (1 = Hit, 0 = Not a Hit)  

---
| Type | Models |
|------|--------|
| **ML** | Logistic Regression, Decision Tree, Random Forest, XGBoost, LightGBM, SVM, KNN |
| **ANN** | Shallow, Deep, Dropout, BatchNorm, Best (Dropout+BN) |
---

## 📦 Cell 1 — Imports & Setup

In [1]:
import os, pickle, warnings, json
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics         import (accuracy_score, precision_score, recall_score,
                                     f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.inspection      import permutation_importance
import xgboost  as xgb
import lightgbm as lgb

import tensorflow as tf
from tensorflow.keras.models    import Sequential
from tensorflow.keras.layers    import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

tf.random.set_seed(42)
np.random.seed(42)

# ── Dark plot theme ───────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor" : "#0f1117",
    "axes.facecolor"   : "#1e2130",
    "axes.edgecolor"   : "#444",
    "axes.labelcolor"  : "#ccc",
    "xtick.color"      : "#aaa",
    "ytick.color"      : "#aaa",
    "text.color"       : "#eee",
    "grid.color"       : "#2a2a3e",
    "grid.linestyle"   : "--",
    "axes.grid"        : True,
    "font.family"      : "monospace",
})
PALETTE = ["#00d4ff","#f9ca24","#6bcb77","#f8a5c2","#a29bfe",
           "#fd79a8","#fdcb6e","#74b9ff","#55efc4","#e17055"]

os.makedirs("models", exist_ok=True)
print("TensorFlow :", tf.__version__)
print("XGBoost    :", xgb.__version__)
print("LightGBM   :", lgb.__version__)
print("Setup complete ✓")

TensorFlow : 2.21.0
XGBoost    : 3.2.0
LightGBM   : 4.6.0
Setup complete ✓


## 🎵 Cell 2 — Dataset Creation

In [2]:
# ─────────────────────────────────────────────────────────
# Dataset: Spotify-style song features (12,000 tracks)
# Target : is_hit = 1 if popularity in top 45%
# Popularity is a direct function of audio features
# so models have a real signal to learn from.
# ─────────────────────────────────────────────────────────

DATA_PATH = "data/spotify_songs.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded: {DATA_PATH}")
else:
    os.makedirs("data", exist_ok=True)
    N = 12000; np.random.seed(42)
    genres = ["pop","hip-hop","rock","edm","r&b","latin","indie","classical","jazz","country"]
    keys   = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]
    genre_arr = np.random.choice(genres, N, p=[0.22,0.18,0.15,0.12,0.10,0.08,0.07,0.03,0.03,0.02])
    dance = np.random.beta(5,3,N); energy = np.random.beta(4,3,N)
    valence = np.random.beta(4,4,N); tempo = np.random.normal(122,28,N).clip(50,220)
    loud  = np.random.uniform(-25,0,N); speech = np.random.beta(1.5,10,N)
    acoustic = np.random.beta(2,5,N); instrum = np.random.exponential(0.1,N).clip(0,1)
    liveness = np.random.beta(2,8,N); dur = np.random.normal(210000,60000,N).clip(60000,600000)
    ts = np.random.choice([3,4,5],N,p=[0.05,0.92,0.03])
    key_arr = np.random.choice(keys,N); mode_arr = np.random.choice(["major","minor"],N,p=[0.6,0.4])
    genre_bonus = {"pop":8,"hip-hop":10,"latin":7,"r&b":6,"edm":3,
                   "rock":2,"country":0,"indie":-5,"jazz":-8,"classical":-10}
    pop_score = (45 + 25*dance + 15*energy + 10*valence + 5*(loud+25)/25
                 - 10*acoustic - 15*instrum - 5*np.abs(tempo-120)/70
                 + np.random.normal(0,8,N)
                 + np.array([genre_bonus[g] for g in genre_arr]))
    pop_score = np.clip(pop_score, 0, 100).astype(int)
    threshold = np.percentile(pop_score, 55)
    df = pd.DataFrame({"genre":genre_arr,"danceability":dance,"energy":energy,
        "key":key_arr,"loudness":loud,"mode":mode_arr,"speechiness":speech,
        "acousticness":acoustic,"instrumentalness":instrum,"liveness":liveness,
        "valence":valence,"tempo":tempo,"duration_ms":dur,"time_signature":ts,
        "popularity":pop_score,"is_hit":(pop_score>=threshold).astype(int)})
    df.to_csv(DATA_PATH, index=False)
    print(f"Dataset created & saved → {DATA_PATH}")

print(f"Shape    : {df.shape}")
print(f"Hit rate : {df['is_hit'].mean():.2%}")
df.head()

Dataset created & saved → data/spotify_songs.csv
Shape    : (12000, 16)
Hit rate : 47.79%


,genre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,popularity,is_hit
0,hip-hop,0.529313,0.515153,C,-14.111370,minor,0.199132,0.410189,0.126574,0.538735,0.839371,132.011460,60000.000000,4,78,1
1,jazz,0.704583,0.879963,F,-24.917175,major,0.078012,0.069452,0.202764,0.120325,0.707027,129.010397,141258.849557,4,72,0
2,r&b,0.680500,0.665446,D,-14.993321,minor,0.053121,0.121723,0.087480,0.143252,0.269654,101.495288,291362.375510,4,86,1
3,edm,0.843134,0.597891,D,-9.227882,minor,0.021911,0.358415,0.023250,0.145863,0.758501,72.167022,269178.157012,4,77,1
4,pop,0.833533,0.436201,G,-8.589744,major,0.502713,0.560155,0.088713,0.121157,0.668308,124.935818,115748.254478,4,91,1


## 🔍 Cell 3 — EDA: Overview & Class Balance

In [ ]:
print(f"Rows      : {df.shape[0]:,}")
print(f"Columns   : {df.shape[1]}")
print(f"Missing   : {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print()
df.describe().T.round(3)

## 📊 Cell 4 — EDA: Distributions

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle("🎵 Spotify EDA — Feature Distributions", fontsize=16, color="#00d4ff", y=1.01)
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.4)

# Class balance
ax1 = fig.add_subplot(gs[0, 0])
counts = df["is_hit"].value_counts()
ax1.pie(counts, labels=["Not a Hit","Hit"], autopct="%1.1f%%",
        colors=["#e17055","#6bcb77"], startangle=90,
        textprops={"color":"white","fontsize":9})
ax1.set_title("Class Balance", color="#00d4ff", fontsize=10)

# Genre distribution
ax2 = fig.add_subplot(gs[0, 1:3])
gc  = df["genre"].value_counts()
ax2.bar(gc.index, gc.values, color=PALETTE)
ax2.set_title("Tracks per Genre", color="#00d4ff", fontsize=10)
ax2.tick_params(axis="x", rotation=35)

# Hit rate by genre
ax3 = fig.add_subplot(gs[0, 3])
hbg = df.groupby("genre")["is_hit"].mean().sort_values()
ax3.barh(hbg.index, hbg.values, color="#a29bfe")
ax3.set_title("Hit Rate by Genre", color="#00d4ff", fontsize=10)

# Feature distributions by class
feats = ["danceability","energy","loudness","speechiness","acousticness","valence"]
pos   = [(1,0),(1,1),(1,2),(1,3),(2,0),(2,1)]
for feat, p in zip(feats, pos):
    ax = fig.add_subplot(gs[p])
    ax.hist(df[df["is_hit"]==0][feat], bins=30, alpha=0.7, color="#e17055", label="Not Hit", density=True)
    ax.hist(df[df["is_hit"]==1][feat], bins=30, alpha=0.7, color="#6bcb77", label="Hit",     density=True)
    ax.set_title(feat, color="#00d4ff", fontsize=9)
    ax.legend(fontsize=7, facecolor="#1e2130", labelcolor="white")

# Popularity histogram with threshold
ax_p = fig.add_subplot(gs[2, 2])
ax_p.hist(df["popularity"], bins=40, color="#f9ca24", edgecolor="black", lw=0.3)
ax_p.axvline(df[df["is_hit"]==0]["popularity"].max(), color="red", lw=2, ls="--", label="Hit threshold")
ax_p.set_title("Popularity Score", color="#00d4ff", fontsize=9)
ax_p.legend(fontsize=7, facecolor="#1e2130", labelcolor="white")

# Tempo by class
ax_t = fig.add_subplot(gs[2, 3])
ax_t.hist(df[df["is_hit"]==0]["tempo"], bins=40, alpha=0.7, color="#e17055", density=True, label="Not Hit")
ax_t.hist(df[df["is_hit"]==1]["tempo"], bins=40, alpha=0.7, color="#6bcb77", density=True, label="Hit")
ax_t.set_title("Tempo", color="#00d4ff", fontsize=9)
ax_t.legend(fontsize=7, facecolor="#1e2130", labelcolor="white")

plt.savefig("models/eda_overview.png", dpi=110, bbox_inches="tight", facecolor="#0f1117")
plt.show()

## 🔥 Cell 5 — EDA: Correlation Heatmap

In [ ]:
num_cols = ["danceability","energy","loudness","speechiness","acousticness",
            "instrumentalness","liveness","valence","tempo",
            "duration_ms","time_signature","popularity","is_hit"]
fig, ax = plt.subplots(figsize=(13, 10))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.3,
            annot_kws={"size":8}, cbar_kws={"shrink":0.7})
ax.set_title("Feature Correlation Heatmap", color="#00d4ff", fontsize=14)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("models/eda_correlation.png", dpi=110, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("Top correlations with is_hit:")
print(corr["is_hit"].drop("is_hit").sort_values(key=abs, ascending=False).round(3).to_string())

## 🎸 Cell 6 — EDA: Genre Deep Dive

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Genre Deep Dive", fontsize=14, color="#00d4ff")

# Danceability boxplot by genre
go = df.groupby("genre")["danceability"].median().sort_values(ascending=False).index
bp = axes[0,0].boxplot([df[df["genre"]==g]["danceability"].values for g in go],
                        labels=go, patch_artist=True)
for patch, c in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(c); patch.set_alpha(0.8)
axes[0,0].set_title("Danceability by Genre", color="#00d4ff", fontsize=11)
axes[0,0].tick_params(axis="x", rotation=35)

# Mean energy by genre
ebg = df.groupby("genre")["energy"].mean().sort_values(ascending=False)
axes[0,1].bar(ebg.index, ebg.values, color=PALETTE[:len(ebg)])
axes[0,1].set_title("Mean Energy by Genre", color="#00d4ff", fontsize=11)
axes[0,1].tick_params(axis="x", rotation=35)

# Hit rate by mode
hm = df.groupby("mode")["is_hit"].mean()
axes[1,0].bar(hm.index, hm.values, color=["#a29bfe","#fd79a8"], width=0.4)
axes[1,0].set_title("Hit Rate: Major vs Minor Key", color="#00d4ff", fontsize=11)
for i,(k,v) in enumerate(hm.items()): axes[1,0].text(i,v+0.005,f"{v:.3f}",ha="center",color="white")

# Popularity violin by genre
gs2 = df.groupby("genre")["popularity"].median().sort_values(ascending=False).index.tolist()
vp  = axes[1,1].violinplot([df[df["genre"]==g]["popularity"].values for g in gs2], showmedians=True)
for body, c in zip(vp["bodies"], PALETTE): body.set_facecolor(c); body.set_alpha(0.7)
axes[1,1].set_xticks(range(1,len(gs2)+1)); axes[1,1].set_xticklabels(gs2, rotation=35)
axes[1,1].set_title("Popularity Distribution by Genre", color="#00d4ff", fontsize=11)

plt.tight_layout()
plt.savefig("models/eda_genre.png", dpi=110, bbox_inches="tight", facecolor="#0f1117")
plt.show()

## ⚙️ Cell 7 — Feature Engineering & Preprocessing

In [3]:
df_m = df.copy()

# Encode categoricals
le_genre = LabelEncoder(); le_key = LabelEncoder(); le_mode = LabelEncoder()
df_m["genre_enc"] = le_genre.fit_transform(df_m["genre"])
df_m["key_enc"]   = le_key.fit_transform(df_m["key"])
df_m["mode_enc"]  = le_mode.fit_transform(df_m["mode"])

# Engineered features
df_m["duration_min"]          = df_m["duration_ms"] / 60000
df_m["loudness_norm"]         = df_m["loudness"] + 25          # shift to [0,25]
df_m["dance_energy"]          = df_m["danceability"] * df_m["energy"]   # interaction
df_m["acoustic_instrumental"] = df_m["acousticness"] * df_m["instrumentalness"]
df_m["mood_score"]            = df_m["valence"] * df_m["energy"]        # mood proxy

FEATURES = [
    "danceability","energy","loudness_norm","speechiness","acousticness",
    "instrumentalness","liveness","valence","tempo","duration_min",
    "time_signature","genre_enc","key_enc","mode_enc",
    "dance_energy","acoustic_instrumental","mood_score"
]
TARGET = "is_hit"

X = df_m[FEATURES].values.astype(np.float32)
y = df_m[TARGET].values

# Split: 68% train | 12% val | 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val,  y_train, y_val  = train_test_split(X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train).astype(np.float32)
X_val_sc    = scaler.transform(X_val).astype(np.float32)
X_test_sc   = scaler.transform(X_test).astype(np.float32)

# Save preprocessors
with open("models/scaler.pkl",  "wb") as f: pickle.dump(scaler, f)
with open("models/features.pkl","wb") as f: pickle.dump(FEATURES, f)
with open("models/encoders.pkl","wb") as f:
    pickle.dump({"genre":le_genre,"key":le_key,"mode":le_mode}, f)

print(f"Features   : {len(FEATURES)}")
print(f"Train      : {X_train_sc.shape}")
print(f"Validation : {X_val_sc.shape}")
print(f"Test       : {X_test_sc.shape}")
print("Preprocessors saved ✓")

Features   : 17
Train      : (8160, 17)
Validation : (1440, 17)
Test       : (2400, 17)
Preprocessors saved ✓


## 🤖 Cell 8 — Train All ML Models

In [4]:
def evaluate_ml(model, name):
    """Fit, cross-validate and evaluate an ML model."""
    model.fit(X_train_sc, y_train)
    yp  = model.predict(X_test_sc)
    ypr = model.predict_proba(X_test_sc)[:,1]
    cv  = cross_val_score(model, X_train_sc, y_train,
                          cv=StratifiedKFold(5, shuffle=True, random_state=42),
                          scoring="accuracy", n_jobs=-1)
    res = {
        "accuracy" : round(accuracy_score(y_test, yp), 4),
        "precision": round(precision_score(y_test, yp), 4),
        "recall"   : round(recall_score(y_test, yp), 4),
        "f1"       : round(f1_score(y_test, yp), 4),
        "roc_auc"  : round(roc_auc_score(y_test, ypr), 4),
        "cv_mean"  : round(cv.mean(), 4),
        "cv_std"   : round(cv.std(), 4),
        "y_pred"   : yp, "y_prob": ypr,
    }
    print(f"  {name:<25} Acc={res['accuracy']:.4f}  F1={res['f1']:.4f}  AUC={res['roc_auc']:.4f}  CV={res['cv_mean']:.4f}±{res['cv_std']:.4f}")
    return res

print("Training ML models...")
print("─"*80)

ml_configs = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Decision Tree"      : DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest"      : RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost"            : xgb.XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.1,
                                              random_state=42, eval_metric="logloss", verbosity=0),
    "LightGBM"           : lgb.LGBMClassifier(n_estimators=150, max_depth=5, random_state=42, verbose=-1),
    "SVM"                : SVC(kernel="rbf", probability=True, random_state=42),
    "KNN"                : KNeighborsClassifier(n_neighbors=9, n_jobs=-1),
}

ml_results = {}
for name, model in ml_configs.items():
    ml_results[name] = evaluate_ml(model, name)
    safe = name.replace(" ","_")
    with open(f"models/ml_{safe}.pkl","wb") as f: pickle.dump(model, f)

print("─"*80)
print("All ML models saved to models/ ✓")

Training ML models...
────────────────────────────────────────────────────────────────────────────────
  Logistic Regression       Acc=0.6775  F1=0.6551  AUC=0.7459  CV=0.6667±0.0117
  Decision Tree             Acc=0.7071  F1=0.6950  AUC=0.7574  CV=0.6663±0.0123
  Random Forest             Acc=0.7087  F1=0.6930  AUC=0.7846  CV=0.6966±0.0159
  XGBoost                   Acc=0.7408  F1=0.7319  AUC=0.8188  CV=0.7300±0.0149
  LightGBM                  Acc=0.7450  F1=0.7358  AUC=0.8209  CV=0.7286±0.0148
  SVM                       Acc=0.6971  F1=0.6776  AUC=0.7534  CV=0.6745±0.0109
  KNN                       Acc=0.6421  F1=0.6343  AUC=0.6863  CV=0.6248±0.0089
────────────────────────────────────────────────────────────────────────────────
All ML models saved to models/ ✓


## 🧠 Cell 9 — Build & Train ANN Models

In [5]:
# ─────────────────────────────────────────────────────────────────────
# Five ANN variants to show effect of:
#   1. Shallow vs Deep architecture
#   2. Dropout regularization
#   3. Batch Normalization
#   4. Combined (Dropout + BN) = Best
# ─────────────────────────────────────────────────────────────────────

n_ft = X_train_sc.shape[1]

def build_ann(hidden_sizes, dropout=0.0, batchnorm=False, lr=0.001):
    """
    Build a Sequential ANN.
    hidden_sizes : list of neuron counts per hidden layer
    dropout      : dropout rate (0 = disabled)
    batchnorm    : add BatchNormalization after each hidden layer
    """
    layers = []
    for sz in hidden_sizes:
        layers.append(Dense(sz, activation="relu", kernel_initializer="he_normal"))
        if batchnorm: layers.append(BatchNormalization())
        if dropout > 0: layers.append(Dropout(dropout))
    layers.append(Dense(1, activation="sigmoid"))
    m = Sequential(layers)
    m.build(input_shape=(None, n_ft))
    m.compile(Adam(lr), "binary_crossentropy", metrics=["accuracy"])
    return m

def train_ann(model, name, epochs=150, patience=15):
    cb = EarlyStopping(monitor="val_loss", patience=patience,
                       restore_best_weights=True, verbose=0)
    h = model.fit(X_train_sc, y_train,
                  validation_data=(X_val_sc, y_val),
                  epochs=epochs, batch_size=64,
                  callbacks=[cb], verbose=0)
    ypr = model.predict(X_test_sc, verbose=0).flatten()
    yp  = (ypr >= 0.5).astype(int)
    res = {
        "accuracy" : round(accuracy_score(y_test, yp), 4),
        "precision": round(precision_score(y_test, yp), 4),
        "recall"   : round(recall_score(y_test, yp), 4),
        "f1"       : round(f1_score(y_test, yp), 4),
        "roc_auc"  : round(roc_auc_score(y_test, ypr), 4),
        "epochs"   : len(h.history["loss"]),
        "y_pred": yp, "y_prob": ypr, "history": h.history,
    }
    print(f"  {name:<30} Acc={res['accuracy']:.4f}  F1={res['f1']:.4f}  "
          f"AUC={res['roc_auc']:.4f}  Stopped@Ep={res['epochs']}")
    return res

# ─── Model Specs ──────────────────────────────────────────
ann_specs = [
    # (name,              sizes,          dropout, batchnorm, epochs, patience)
    ("ANN Shallow",       [64, 32],       0.0,     False,     150,    15),
    ("ANN Deep",          [128,64,32,16], 0.0,     False,     150,    15),
    ("ANN + Dropout",     [128, 64, 32],  0.3,     False,     200,    20),
    ("ANN + BatchNorm",   [128, 64, 32],  0.2,     True,      200,    20),
    ("ANN Best (D+BN)",   [256,128,64,32],0.3,     True,      250,    25),
]

print("Training ANN models...")
print("─"*80)

ann_results = {}
ann_models  = {}

for name, sizes, drop, bn, epochs, patience in ann_specs:
    tf.random.set_seed(42); np.random.seed(42)
    m = build_ann(sizes, drop, bn)
    ann_results[name] = train_ann(m, name, epochs, patience)
    ann_models[name]  = m
    safe = name.replace(" ","_").replace("+","plus").replace("(","").replace(")","")
    hist_clean = {k:[float(v) for v in vals] for k,vals in ann_results[name]["history"].items()}
    with open(f"models/ann_{safe}.pkl","wb") as f:
        pickle.dump({"weights":m.get_weights(),"history":hist_clean,"sizes":sizes,"drop":drop,"bn":bn}, f)

print("─"*80)
print("All ANN models saved ✓")

Training ANN models...
────────────────────────────────────────────────────────────────────────────────
  ANN Shallow                    Acc=0.6946  F1=0.6714  AUC=0.7579  Stopped@Ep=26
  ANN Deep                       Acc=0.6729  F1=0.6503  AUC=0.7460  Stopped@Ep=16
  ANN + Dropout                  Acc=0.6933  F1=0.6921  AUC=0.7633  Stopped@Ep=72
  ANN + BatchNorm                Acc=0.6937  F1=0.6852  AUC=0.7593  Stopped@Ep=40
  ANN Best (D+BN)                Acc=0.7025  F1=0.6938  AUC=0.7664  Stopped@Ep=52
────────────────────────────────────────────────────────────────────────────────
All ANN models saved ✓


## 📐 Cell 10 — ANN Architecture Diagrams

In [ ]:
def draw_ann(layer_sizes, dropout=0.0, title="ANN", ax=None):
    """Draw a clean ANN architecture diagram using Matplotlib."""
    if ax is None: fig, ax = plt.subplots(figsize=(12,5))
    n_layers = len(layer_sizes); node_r = 0.22
    max_n    = max(layer_sizes)
    layer_colors = ["#3498db"] + ["#2ecc71"]*(n_layers-2) + ["#f39c12"]
    positions = []
    for li, n in enumerate(layer_sizes):
        x   = li * 2.5
        ys  = np.linspace(-(n-1)/2, (n-1)/2, n)
        lp  = []
        for yi in ys:
            ax.add_patch(plt.Circle((x,yi), node_r,
                         color=layer_colors[li], ec="white", lw=1.2, zorder=3))
            lp.append((x,yi))
        positions.append(lp)
        label = "Input" if li==0 else ("Output" if li==n_layers-1 else f"Hidden {li}")
        dtxt  = f"
+Drop({dropout})" if dropout>0 and 0<li<n_layers-1 else ""
        ax.text(x, -(max_n/2)-0.9, f"{label}
({n}){dtxt}",
                ha="center", va="top", fontsize=8, color="white")
    for li in range(n_layers-1):  # draw edges (sampled)
        for (x1,y1) in positions[li][:6]:
            for (x2,y2) in positions[li+1][:6]:
                ax.plot([x1+node_r,x2-node_r],[y1,y2],"white",alpha=0.08,lw=0.5,zorder=1)
    ax.set_xlim(-0.8,(n_layers-1)*2.5+0.8); ax.set_ylim(-(max_n/2)-2, max_n/2+1)
    ax.set_facecolor("#1e2130"); ax.set_title(title,color="#00d4ff",fontsize=10,fontweight="bold")
    ax.axis("off")

specs = [
    ([30,64,32,1],       0.0,  "ANN Shallow
[64→32]"),
    ([30,128,64,32,16,1],0.0,  "ANN Deep
[128→64→32→16]"),
    ([30,128,64,32,1],   0.3,  "ANN + Dropout(0.3)
[128→64→32]"),
    ([30,128,64,32,1],   0.2,  "ANN + BatchNorm + Drop
[128→64→32]"),
    ([30,256,128,64,32,1],0.3, "ANN Best (D+BN)
[256→128→64→32]"),
]

fig, axes = plt.subplots(2,3, figsize=(18,10), facecolor="#0f1117")
fig.suptitle("ANN Architecture Comparison", fontsize=16, color="#00d4ff")
axes = axes.flatten()
for i,(ls,dr,title) in enumerate(specs):
    draw_ann(ls, dr, title, axes[i])
axes[-1].axis("off")
plt.tight_layout()
plt.savefig("models/all_architectures.png", dpi=110, bbox_inches="tight", facecolor="#0f1117")
plt.show()

## 📈 Cell 11 — ANN Training Curves

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(16, 22))
fig.suptitle("ANN Training Curves — Loss & Accuracy", fontsize=14, color="#00d4ff", y=1.01)
colors = ["#00d4ff","#6bcb77","#f9ca24","#a29bfe","#fd79a8"]

for i, (name, res) in enumerate(ann_results.items()):
    hist = res["history"]
    ep   = range(1, len(hist["loss"]) + 1)
    c    = colors[i]; stop = res["epochs"]

    axes[i,0].plot(ep, hist["loss"],          color=c, lw=2,       label="Train Loss")
    axes[i,0].plot(ep, hist["val_loss"],      color=c, lw=2, ls="--", label="Val Loss")
    axes[i,0].axvline(stop, color="red", ls=":", lw=1.5, label=f"Stop@{stop}")
    axes[i,0].set_title(f"{name} — Loss", color="#00d4ff", fontsize=10)
    axes[i,0].legend(fontsize=8, facecolor="#1e2130", labelcolor="white")

    axes[i,1].plot(ep, hist["accuracy"],      color=c, lw=2,       label="Train Acc")
    axes[i,1].plot(ep, hist["val_accuracy"],  color=c, lw=2, ls="--", label="Val Acc")
    axes[i,1].axvline(stop, color="red", ls=":", lw=1.5)
    axes[i,1].set_title(f"{name} — Accuracy", color="#00d4ff", fontsize=10)
    axes[i,1].legend(fontsize=8, facecolor="#1e2130", labelcolor="white")

plt.tight_layout()
plt.savefig("models/ann_training_curves.png", dpi=110, bbox_inches="tight", facecolor="#0f1117")
plt.show()

## ⚖️ Cell 12 — ML vs ANN Comparison Dashboard

In [ ]:
# Merge all results into one dict
all_res = {f"ML: {k}":v  for k,v in ml_results.items()}
all_res.update({f"NN: {k}":v for k,v in ann_results.items()})

mdf = pd.DataFrame({n: {"Accuracy":r["accuracy"],"Precision":r["precision"],
                          "Recall":r["recall"],"F1 Score":r["f1"],"ROC-AUC":r["roc_auc"]}
                     for n,r in all_res.items()}).T.sort_values("F1 Score",ascending=False)

names_s = [n.replace("ML: ","").replace("NN: ","") for n in mdf.index]
b_clrs  = ["#a29bfe" if "NN:" in n else "#6bcb77" for n in mdf.index]

fig = plt.figure(figsize=(20, 15))
fig.suptitle("🎵 ML vs ANN — Complete Comparison", fontsize=16, color="#00d4ff", y=1.01)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)

# 1. Accuracy bar chart
ax1 = fig.add_subplot(gs[0,:])
bars = ax1.bar(names_s, mdf["Accuracy"], color=b_clrs, edgecolor="white", lw=0.5)
for bar,v in zip(bars,mdf["Accuracy"]):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003, f"{v:.3f}",
             ha="center", va="bottom", fontsize=8, color="white")
ax1.set_title("Test Accuracy (Green=ML  Purple=ANN)", color="#00d4ff", fontsize=13)
ax1.set_ylim(mdf["Accuracy"].min()-0.05, 1.0); ax1.tick_params(axis="x",rotation=35)

# 2. F1 horizontal bar
ax2 = fig.add_subplot(gs[1,0])
ax2.barh(names_s[::-1], mdf["F1 Score"][::-1], color=b_clrs[::-1])
ax2.set_title("F1 Score", color="#00d4ff", fontsize=11)

# 3. ROC-AUC horizontal bar
ax3 = fig.add_subplot(gs[1,1])
ax3.barh(names_s[::-1], mdf["ROC-AUC"][::-1], color="#fdcb6e")
ax3.set_title("ROC-AUC", color="#00d4ff", fontsize=11)

# 4. Precision vs Recall scatter
ax4 = fig.add_subplot(gs[1,2])
for n,r in all_res.items():
    c = "#a29bfe" if "NN:" in n else "#6bcb77"
    ax4.scatter(r["recall"], r["precision"], color=c, s=90, zorder=3)
    ax4.annotate(n.replace("ML: ","").replace("NN: ","")[:10],
                 (r["recall"],r["precision"]), fontsize=6, color="#bbb",
                 xytext=(3,3), textcoords="offset points")
ax4.set_xlabel("Recall"); ax4.set_ylabel("Precision")
ax4.set_title("Precision vs Recall", color="#00d4ff", fontsize=11)

# 5. ROC curves: top 3 ML + best ANN
ax5 = fig.add_subplot(gs[2,0:2])
roc_models = [
    ("ml_LightGBM",    "LightGBM",     "#6bcb77"),
    ("ml_XGBoost",     "XGBoost",      "#f9ca24"),
    ("ml_Random_Forest","Random Forest","#00d4ff"),
]
for fname, label, color in roc_models:
    with open(f"models/{fname}.pkl","rb") as f: m = pickle.load(f)
    ypr = m.predict_proba(X_test_sc)[:,1]
    fpr, tpr, _ = roc_curve(y_test, ypr)
    ax5.plot(fpr, tpr, color=color, lw=2, label=f"{label} (AUC={roc_auc_score(y_test,ypr):.3f})")
# Best ANN
best_ann_name = max(ann_results, key=lambda n: ann_results[n]["roc_auc"])
ypr_ann = ann_results[best_ann_name]["y_prob"]
fpr,tpr,_ = roc_curve(y_test, ypr_ann)
ax5.plot(fpr,tpr,color="#a29bfe",lw=2,label=f"Best ANN: {best_ann_name} (AUC={roc_auc_score(y_test,ypr_ann):.3f})")
ax5.plot([0,1],[0,1],"k--",lw=1,alpha=0.4)
ax5.set_xlabel("FPR"); ax5.set_ylabel("TPR")
ax5.set_title("ROC Curves", color="#00d4ff", fontsize=11)
ax5.legend(facecolor="#1e2130",labelcolor="white",fontsize=9)

# 6. Radar chart
ax6 = fig.add_subplot(gs[2,2], polar=True)
mkeys = ["accuracy","precision","recall","f1","roc_auc"]
mlbls = ["Accuracy","Precision","Recall","F1","AUC"]
angs  = np.linspace(0,2*np.pi,len(mkeys),endpoint=False).tolist()+[0]
best_ml_name = max(ml_results, key=lambda n: ml_results[n]["roc_auc"])
for src,name,color,lbl in [
    (ml_results,  best_ml_name,  "#6bcb77", f"ML: {best_ml_name}"),
    (ann_results, best_ann_name, "#a29bfe", f"ANN: {best_ann_name}"),
]:
    vals = [src[name][k] for k in mkeys]+[src[name][mkeys[0]]]
    ax6.plot(angs,vals,color=color,lw=2,label=lbl)
    ax6.fill(angs,vals,color=color,alpha=0.15)
ax6.set_xticks(angs[:-1]); ax6.set_xticklabels(mlbls,fontsize=8)
ax6.set_ylim(0.5,1.0)
ax6.set_title("Best ML vs ANN
Radar",color="#00d4ff",fontsize=10)
ax6.legend(facecolor="#1e2130",labelcolor="white",fontsize=8,loc="upper right",bbox_to_anchor=(1.4,1.15))

plt.savefig("models/comparison_dashboard.png", dpi=110, bbox_inches="tight", facecolor="#0f1117")
plt.show()

## 🔲 Cell 13 — Confusion Matrices (All Models)

In [ ]:
n_models = len(all_res); ncols=4; nrows=(n_models+ncols-1)//ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*3.5))
fig.suptitle("Confusion Matrices — All Models", fontsize=14, color="#00d4ff")
axes = axes.flatten()

for i,(name,res) in enumerate(all_res.items()):
    cm   = confusion_matrix(y_test, res["y_pred"])
    cmap = "Purples" if "NN:" in name else "Greens"
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=["Not Hit","Hit"], yticklabels=["Not Hit","Hit"],
                ax=axes[i], cbar=False, linewidths=0.5)
    axes[i].set_title(
        f"{name.replace(chr(77)+chr(76)+chr(58)+chr(32),"").replace(chr(78)+chr(78)+chr(58)+chr(32),"")}
Acc={res["accuracy"]:.3f}",
        color="#00d4ff",fontsize=8)
    axes[i].tick_params(colors="white",labelsize=8)

for j in range(i+1,len(axes)): axes[j].axis("off")
plt.tight_layout()
plt.savefig("models/all_confusion_matrices.png", dpi=110, bbox_inches="tight", facecolor="#0f1117")
plt.show()

## 🌲 Cell 14 — Feature Importance

In [ ]:
with open("models/ml_Random_Forest.pkl","rb") as f: rf_model = pickle.load(f)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Feature Importance Analysis", fontsize=14, color="#00d4ff")

# Built-in RF importance
imp = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
axes[0].barh(imp.index, imp.values, color="#6bcb77")
axes[0].set_title("Random Forest — Built-in Importance", color="#00d4ff", fontsize=11)

# Permutation importance on test set
perm = permutation_importance(rf_model, X_test_sc, y_test, n_repeats=8, random_state=42, n_jobs=-1)
pimp = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=True)
axes[1].barh(pimp.index, pimp.values, color="#f9ca24")
axes[1].set_title("Permutation Importance (Test Set)", color="#00d4ff", fontsize=11)

plt.tight_layout()
plt.savefig("models/feature_importance.png", dpi=110, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("Top 5 features (permutation):")
print(pimp.sort_values(ascending=False).head(5).round(4).to_string())

## 📋 Cell 15 — Final Leaderboard

In [ ]:
final = pd.DataFrame({
    n: {"Type": "ANN" if "NN:" in n else "ML",
        "Model": n.replace("ML: ","").replace("NN: ",""),
        "Accuracy" : r["accuracy"],
        "Precision": r["precision"],
        "Recall"   : r["recall"],
        "F1 Score" : r["f1"],
        "ROC-AUC"  : r["roc_auc"]}
    for n,r in all_res.items()
}).T.sort_values("F1 Score", ascending=False)
final.index = range(1, len(final)+1)

print("="*75)
print("  FINAL LEADERBOARD")
print("="*75)
print(final[["Type","Model","Accuracy","Precision","Recall","F1 Score","ROC-AUC"]].to_string())
print()
best_all = mdf["F1 Score"].idxmax()
best_ml  = mdf[mdf.index.str.startswith("ML:")]["F1 Score"].idxmax()
best_ann = mdf[mdf.index.str.startswith("NN:")]["F1 Score"].idxmax()
print(f"  Winner  : {best_all}")
print(f"  Best ML : {best_ml}")
print(f"  Best ANN: {best_ann}")
print()
print("Saved models:")
for fname in sorted(os.listdir("models")):
    if fname.endswith(".pkl"): print(f"  models/{fname}")

## 💾 Cell 16 — Load Saved Model & Predict

In [ ]:
# Demo: Load LightGBM (best ML) and make a live prediction
with open("models/ml_LightGBM.pkl",  "rb") as f: lgbm_loaded  = pickle.load(f)
with open("models/scaler.pkl",        "rb") as f: sc_loaded    = pickle.load(f)
with open("models/features.pkl",      "rb") as f: feat_loaded  = pickle.load(f)

# Sample song: high-energy pop with great danceability
sample = {
    "danceability":0.87, "energy":0.92, "loudness_norm":21.0,
    "speechiness":0.06, "acousticness":0.04, "instrumentalness":0.00,
    "liveness":0.09, "valence":0.80, "tempo":126.0,
    "duration_min":3.4, "time_signature":4,
    "genre_enc":1,     # hip-hop (encoded)
    "key_enc":5, "mode_enc":0,
    "dance_energy":0.87*0.92, "acoustic_instrumental":0.04*0.00, "mood_score":0.80*0.92
}

arr = np.array([[sample[f] for f in feat_loaded]]).astype(np.float32)
arr_sc = sc_loaded.transform(arr)

pred  = lgbm_loaded.predict(arr_sc)[0]
proba = lgbm_loaded.predict_proba(arr_sc)[0]

print("Song Profile:")
print("  Genre: hip-hop | Danceability: 0.87 | Energy: 0.92 | Valence: 0.80")
print()
print("LightGBM Prediction:")
print(f"  → {chr(127926)+" HIT!" if pred==1 else chr(128201)+" Not a Hit"}")
print(f"  Hit probability     : {proba[1]:.4f}")
print(f"  Not Hit probability : {proba[0]:.4f}")